# WSI – Ćwiczenie 5: Sztuczne Sieci Neuronowe

**Autorzy:** Mikołaj Wróbel, Kacper Maciejko  
**Data:** 2026  

---

## 1. Opis implementacji

### Architektura sieci (MLP)
- Implementacja oparta wyłącznie o **numpy** (bez PyTorch / TensorFlow).
- Sieć składa się z obiektów: `Model → [Layer] → [Node]`.
- Każdy `Node` przechowuje wektor wag i bias.
- Konfigurowalny: dowolna liczba warstw ukrytych i dowolna liczba neuronów.
- Funkcje aktywacji w warstwach ukrytych: **ReLU** (domyślna) lub **Sigmoid**.
- Warstwa wyjściowa: **Softmax**.
- Funkcja straty: **entropia krzyżowa** (cross-entropy).

### Algorytm optymalizacji
- **Mini-batch SGD** z propagacją wsteczną.
- Konfigurowalny współczynnik uczenia η i rozmiar mini-batcha.

### Normalizacja danych
- Piksele w zakresie 0–16 podzielone przez 16 → zakres **[0, 1]**.

### Podział danych (stratified)
- Trenujący: **60%**, Walidacyjny: **20%**, Testowy: **20%**.

In [1]:
# ── Setup ─────────────────────────────────────────────────────────────
from pathlib import Path
import sys
import numpy as np

ROOT_DIR = Path('').resolve().parents[0]   # adjust if running from a subdirectory
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from Code.Import_training_data.import_traing_data import DataLoader
from Code.train.model import Model
from Code.train.training import TrainModel

loader = DataLoader()
X_train, y_train = loader.get_train_data()
X_val,   y_val   = loader.get_val_data()
X_test,  y_test  = loader.get_test_data()

print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')
print(f'Classes: {np.unique(y_train)}')


Train: (1078, 64)  Val: (359, 64)  Test: (360, 64)
Classes: [0 1 2 3 4 5 6 7 8 9]


---
## 2. Eksperymenty porównawcze

Przetestowano **3 architektury × 2 współczynniki uczenia × 2 rozmiary mini-batcha**.

| # | Architektura | η | Batch | Test acc |
|---|---|---|---|---|
| 1 | 64→64→10 | 0.01 | pełny | *(wypełnić)* |
| 2 | 64→128→10 | 0.01 | pełny | *(wypełnić)* |
| 3 | 64→128→64→10 | 0.01 | pełny | *(wypełnić)* |
| 4 | 64→128→10 | 0.05 | pełny | *(wypełnić)* |
| 5 | 64→128→10 | 0.01 | 32 | *(wypełnić)* |
| 6 | 64→128→10 | 0.05 | 32 | *(wypełnić)* |

In [2]:
# ── Helper: train one config and return (trainer, test_acc) ───────────
def run_experiment(layer_sizes, activations, lr, epochs, batch_size=None):
    model   = Model(layer_sizes, activations)
    trainer = TrainModel(model, learning_rate=lr)

    if batch_size is None:
        # Full-batch
        trainer.train(X_train, y_train, X_val, y_val, epochs=epochs, verbose=False)
    else:
        # Mini-batch SGD
        trainer.loss_history    = []
        trainer.val_acc_history = []
        n = X_train.shape[0]
        for epoch in range(1, epochs + 1):
            idx = np.random.permutation(n)
            for start in range(0, n, batch_size):
                batch = idx[start:start + batch_size]
                trainer.gradient_descent(X_train[batch], y_train[batch])
            if epoch % 5 == 0:
                loss    = trainer._cross_entropy_loss(trainer._forward(X_train), y_train)
                val_acc = trainer.accuracy(X_val, y_val)
                trainer.loss_history.append(round(float(loss), 5))
                trainer.val_acc_history.append(round(float(val_acc), 5))

    test_acc = trainer.accuracy(X_test, y_test)
    return trainer, test_acc

In [3]:
# ── Run all experiments ───────────────────────────────────────────────
EPOCHS = 200

experiments = [
    {'name': '64→64→10,        η=0.01, full',   'sizes': [64, 64, 10],       'act': ['relu','softmax'],        'lr': 0.01, 'batch': None},
    {'name': '64→128→10,       η=0.01, full',   'sizes': [64, 128, 10],      'act': ['relu','softmax'],        'lr': 0.01, 'batch': None},
    {'name': '64→128→64→10,    η=0.01, full',   'sizes': [64, 128, 64, 10],  'act': ['relu','relu','softmax'], 'lr': 0.01, 'batch': None},
    {'name': '64→128→10,       η=0.05, full',   'sizes': [64, 128, 10],      'act': ['relu','softmax'],        'lr': 0.05, 'batch': None},
    {'name': '64→128→10,       η=0.01, b=32',   'sizes': [64, 128, 10],      'act': ['relu','softmax'],        'lr': 0.01, 'batch': 32},
    {'name': '64→128→10,       η=0.05, b=32',   'sizes': [64, 128, 10],      'act': ['relu','softmax'],        'lr': 0.05, 'batch': 32},
]

results = []
for exp in experiments:
    trainer, test_acc = run_experiment(
        exp['sizes'], exp['act'], exp['lr'], EPOCHS, exp['batch']
    )
    exp['trainer']  = trainer
    exp['test_acc'] = test_acc
    results.append(exp)
    print(f"{exp['name']:<40}  test acc = {test_acc:.4f}")

64→64→10,        η=0.01, full             test acc = 0.6417
64→128→10,       η=0.01, full             test acc = 0.6806
64→128→64→10,    η=0.01, full             test acc = 0.8167
64→128→10,       η=0.05, full             test acc = 0.8444
64→128→10,       η=0.01, b=32             test acc = 0.8833
64→128→10,       η=0.05, b=32             test acc = 0.9333


---
## 3. Krzywe uczenia

In [4]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Training Loss', 'Validation Accuracy'),
)

colors = ['#636EFA','#EF553B','#00CC96','#AB63FA','#FFA15A','#19D3F3']

for i, exp in enumerate(results):
    t = exp['trainer']
    xs = [j * 5 for j in range(1, len(t.loss_history) + 1)]
    c  = colors[i % len(colors)]

    fig.add_trace(go.Scatter(x=xs, y=t.loss_history,
                             name=exp['name'], line=dict(color=c),
                             legendgroup=exp['name']),
                  row=1, col=1)
    fig.add_trace(go.Scatter(x=xs, y=t.val_acc_history,
                             name=exp['name'], line=dict(color=c),
                             legendgroup=exp['name'], showlegend=False),
                  row=1, col=2)

fig.update_xaxes(title_text='Epoch', gridcolor='#333')
fig.update_yaxes(gridcolor='#333')
fig.update_layout(
    title='Learning Curves – all experiments',
    plot_bgcolor='#1e1e2e', paper_bgcolor='#1e1e2e',
    font=dict(color='white'), height=450,
)
fig.show()

---
## 3c. Optymalizacja Bayesowska

Zamiast ręcznie dobierać hiperparametry, używamy **Bayesian Optimisation** z procesem Gaussa (GP) jako modelem zastępczym i funkcją akwizycji UCB.  
Optimizer automatycznie wybiera obiecujące konfiguracje, trenuje je i zwraca najlepszy model — bez ponownego trenowania.


In [5]:
from Code.Optymalize_Model_Size.Bayes_optymalization import BayesOptimizer

# Run Bayesian optimisation over the architecture / hyperparameter space
optimizer = BayesOptimizer(
    X_train, y_train,
    X_val,   y_val,
    layers_range    = (1, 4),
    nodes_range     = (16, 128),
    lr_range        = (1e-3, 0.1),
    epochs_range    = (50, 200),
    n_random_starts = 3,
    n_iterations    = 30,
)
bayes_config, bayes_val_acc, bayes_trainer = optimizer.optimise()

# Reconstruct layer sizes and activation list from the found config
_input_size  = X_train.shape[1]   # 64
_output_size = 10

bayes_sizes = (
    [_input_size]
    + [bayes_config['nodes']] * bayes_config['num_layers']
    + [_output_size]
)
bayes_acts = ['relu'] * bayes_config['num_layers'] + ['softmax']

# Evaluate on held-out test set (model is already trained — no re-training)
bayes_test_acc = bayes_trainer.accuracy(X_test, y_test)
print(f"\nBayes best → layers={bayes_config['num_layers']}, "
      f"nodes={bayes_config['nodes']}, "
      f"lr={bayes_config['lr']:.5f}, epochs={bayes_config['epochs']}")
print(f"Val accuracy:  {bayes_val_acc:.4f}")
print(f"Test accuracy: {bayes_test_acc:.4f}")

# Build a 'best' dict compatible with the visualization cell below
best = {
    'name'     : (f"Bayes: {bayes_config['num_layers']}×{bayes_config['nodes']} "
                  f"lr={bayes_config['lr']:.4f} ep={bayes_config['epochs']}"),
    'sizes'    : bayes_sizes,
    'act'      : bayes_acts,
    'trainer'  : bayes_trainer,
    'test_acc' : bayes_test_acc,
}


  Bayesian Optimisation — searching for best architecture

[ 1/30] random | layers=2, nodes=85, lr=0.0359, epochs=153
  → Val accuracy: 0.8468

[ 2/30] random | layers=3, nodes=55, lr=0.0504, epochs=168
  → Val accuracy: 0.0919

[ 3/30] random | layers=1, nodes=99, lr=0.0819, epochs=109
  → Val accuracy: 0.8552

[ 4/30] GP-guided | layers=2, nodes=122, lr=0.0079, epochs=70
  → Val accuracy: 0.8273

[ 5/30] GP-guided | layers=1, nodes=19, lr=0.0108, epochs=97
  → Val accuracy: 0.1811

[ 6/30] GP-guided | layers=1, nodes=118, lr=0.0305, epochs=196
  → Val accuracy: 0.8524

[ 7/30] GP-guided | layers=2, nodes=124, lr=0.0956, epochs=181
  → Val accuracy: 0.7967

[ 8/30] GP-guided | layers=3, nodes=125, lr=0.0722, epochs=88
  → Val accuracy: 0.1003

[ 9/30] GP-guided | layers=1, nodes=80, lr=0.0958, epochs=189
  → Val accuracy: 0.8858

[10/30] GP-guided | layers=3, nodes=125, lr=0.0116, epochs=195
  → Val accuracy: 0.3454

[11/30] GP-guided | layers=1, nodes=110, lr=0.0417, epochs=134
  → V

---
## 3b. Wizualizacje interaktywne

Poniższe wykresy otwierają się w przeglądarce:
- **Architektura 3D** – obracaj myszą, przewijaj aby przybliżyć.
- **Postęp trenowania** – najedź kursorem na punkt aby zobaczyć dokładne wartości.


In [6]:
from Code.Visualization.network_visualization import show_architecture_3d, show_training_progress

# 'best' is set by the Bayesian optimisation cell above

# ── 3D architecture of the best experiment ────────────────────────────
best_sizes = best['sizes']
best_acts  = best['act']

print(f"Showing 3D architecture for: {best['name']}")
show_architecture_3d(best_sizes, best_acts)

# ── Animated training progress of the best experiment ─────────────────
print(f"Showing training progress for: {best['name']}")
show_training_progress(best['trainer'].loss_history, best['trainer'].val_acc_history)


Showing 3D architecture for: Bayes: 1×115 lr=0.0628 ep=188


Showing training progress for: Bayes: 1×115 lr=0.0628 ep=188


In [7]:
from Code.Visualization.network_visualization import show_architecture

# ── Compare training-progress curves for every experiment in one chart ─
# (inline version – does not open a new browser tab per experiment)
fig_all = go.Figure()
colors  = ['#636EFA','#EF553B','#00CC96','#AB63FA','#FFA15A','#19D3F3']

for i, exp in enumerate(results):
    t  = exp['trainer']
    xs = [j * 5 for j in range(1, len(t.val_acc_history) + 1)]
    c  = colors[i % len(colors)]

    # Animate using frames: each frame adds one more data point
    fig_all.add_trace(go.Scatter(
        x=xs, y=t.val_acc_history,
        mode='lines+markers',
        name=exp['name'],
        line=dict(color=c, width=2),
        marker=dict(size=5),
        hovertemplate='Epoch %{x}<br>Val Acc: %{y:.4f}<extra></extra>',
    ))

fig_all.update_layout(
    title='Validation Accuracy – all experiments',
    xaxis_title='Epoch',
    yaxis_title='Val Accuracy',
    yaxis=dict(range=[0, 1.05], gridcolor='#333'),
    xaxis=dict(gridcolor='#333'),
    plot_bgcolor='#1e1e2e',
    paper_bgcolor='#1e1e2e',
    font=dict(color='white'),
    height=420,
    legend=dict(font=dict(size=10)),
    updatemenus=[dict(
        type='buttons',
        showactive=False,
        y=1.15, x=0.5, xanchor='center',
        buttons=[
            dict(
                label='▶  Play',
                method='animate',
                args=[None, dict(frame=dict(duration=80, redraw=True), fromcurrent=True)],
            ),
            dict(
                label='⏸  Pause',
                method='animate',
                args=[[None], dict(frame=dict(duration=0), mode='immediate')],
            ),
        ],
    )],
)

# Build animation frames – each frame reveals one more epoch point
max_points = max(len(e['trainer'].val_acc_history) for e in results)
frames = []
for frame_idx in range(1, max_points + 1):
    frame_traces = []
    for i, exp in enumerate(results):
        t  = exp['trainer']
        xs = [j * 5 for j in range(1, len(t.val_acc_history) + 1)]
        visible_x = xs[:frame_idx]
        visible_y = t.val_acc_history[:frame_idx]
        frame_traces.append(go.Scatter(x=visible_x, y=visible_y))
    frames.append(go.Frame(data=frame_traces, name=str(frame_idx)))

fig_all.frames = frames
fig_all.show()


---
## 4. Najlepsza konfiguracja – macierz pomyłek

In [8]:
# Pick the best experiment by test accuracy
best = max(results, key=lambda e: e['test_acc'])
print(f"Best config : {best['name']}")
print(f"Test accuracy: {best['test_acc']:.4f}")

# Confusion matrix
best_trainer = best['trainer']
y_pred_proba = best_trainer._forward(X_test)
y_pred       = np.argmax(y_pred_proba, axis=1)

n_classes = 10
cm = np.zeros((n_classes, n_classes), dtype=int)
for true, pred in zip(y_test, y_pred):
    cm[true][pred] += 1

fig_cm = go.Figure(go.Heatmap(
    z=cm,
    x=[str(i) for i in range(n_classes)],
    y=[str(i) for i in range(n_classes)],
    colorscale='Blues',
    text=cm, texttemplate='%{text}',
    hovertemplate='True: %{y}<br>Pred: %{x}<br>Count: %{z}<extra></extra>',
))
fig_cm.update_layout(
    title=f'Confusion Matrix – {best["name"]}',
    xaxis_title='Predicted', yaxis_title='True',
    plot_bgcolor='#1e1e2e', paper_bgcolor='#1e1e2e',
    font=dict(color='white'), height=500,
    yaxis=dict(autorange='reversed'),
)
fig_cm.show()

Best config : 64→128→10,       η=0.05, b=32
Test accuracy: 0.9333


---
## 5. Wyniki w formie tabeli

In [9]:
import pandas as pd

df = pd.DataFrame([
    {
        'Konfiguracja': e['name'],
        'η':            e['lr'],
        'Batch':        e['batch'] if e['batch'] else 'full',
        'Test Acc':     f"{e['test_acc']:.4f}",
        'Best Val Acc': f"{max(e['trainer'].val_acc_history):.4f}" if e['trainer'].val_acc_history else 'N/A',
    }
    for e in results
])
df

,Konfiguracja,η,Batch,Test Acc,Best Val Acc
0,"64→64→10, η=0.01, full",0.01,full,0.6417,0.6546
1,"64→128→10, η=0.01, full",0.01,full,0.6806,0.7187
2,"64→128→64→10, η=0.01, full",0.01,full,0.8167,0.8440
3,"64→128→10, η=0.05, full",0.05,full,0.8444,0.8719
4,"64→128→10, η=0.01, b=32",0.01,32,0.8833,0.9443
5,"64→128→10, η=0.05, b=32",0.05,32,0.9333,0.9387


---
## 6. Wnioski

*(Wypełnij po przeprowadzeniu eksperymentów)*

- **Wpływ architektury:** ...
- **Wpływ współczynnika uczenia η:** ...
- **Wpływ rozmiaru mini-batcha:** ...
- **Najlepsza konfiguracja:** ...
- **Obserwacje z macierzy pomyłek:** ...

---
## 7. Interaktywny rysunek

Kliknij komórki w siatce **8×8** (taki sam rozmiar jak dane treningowe), aby zapalić piksele, a następnie kliknij **▶ Predict**.  
Sieć zwróci słupkowy wykres prawdopodobieństwa dla każdej z 10 cyfr.


In [ ]:
import ipywidgets as widgets
from IPython.display import display
import numpy as np
import plotly.graph_objects as go

# ── 8×8 toggle-button grid ───────────────────────────────────────────
# The sklearn digits dataset is 8×8 pixels — this grid matches exactly,
# so the model receives input at the same resolution it was trained on.
GRID = 8
_ON  = '#1a1a2e'   # dark = pixel lit
_OFF = '#f5f5f5'   # light = pixel off

_btns = [[
    widgets.ToggleButton(
        value=False,
        layout=widgets.Layout(width='36px', height='36px', margin='1px', padding='0px'),
    )
    for _ in range(GRID)
] for _ in range(GRID)]

# Keep button colour in sync with toggle state
for row in _btns:
    for btn in row:
        btn.style.button_color = _OFF
        def _sync_color(change, b=btn):
            b.style.button_color = _ON if change['new'] else _OFF
        btn.observe(_sync_color, names='value')

# ── Controls ─────────────────────────────────────────────────────────
_predict_btn = widgets.Button(
    description='▶ Predict',
    button_style='primary',
    layout=widgets.Layout(width='110px', margin='10px 8px 0 0'),
)
_clear_btn = widgets.Button(
    description='✕ Clear',
    layout=widgets.Layout(width='90px', margin='10px 0 0 0'),
)
_pred_out = widgets.Output()

# ── Callbacks ────────────────────────────────────────────────────────
def _on_predict(_):
    # Read 8×8 grid: dark cell → 16, light cell → 0  (sklearn digits scale)
    px = np.array(
        [[16.0 if _btns[r][c].value else 0.0 for c in range(GRID)] for r in range(GRID)],
        dtype=float,
    ).flatten()

    x  = (px / 16.0).reshape(1, -1)          # normalize to [0, 1]
    pr = best['trainer']._forward(x)[0]       # forward pass through best model
    p  = int(np.argmax(pr))

    with _pred_out:
        _pred_out.clear_output(wait=True)
        fig = go.Figure(go.Bar(
            x    = list(range(10)),
            y    = pr.tolist(),
            text = [f'{v:.0%}' for v in pr],
            textposition = 'outside',
            marker_color = ['#00CC96' if i == p else '#636EFA' for i in range(10)],
        ))
        fig.update_layout(
            title      = f'Prediction: <b>{p}</b>  ({pr[p]:.1%} confidence)',
            xaxis      = dict(title='Digit', tickmode='array', tickvals=list(range(10))),
            yaxis      = dict(title='Probability', range=[0, 1.2], gridcolor='#333'),
            plot_bgcolor  = '#1e1e2e',
            paper_bgcolor = '#1e1e2e',
            font          = dict(color='white'),
            height        = 300,
            margin        = dict(t=55, b=30),
        )
        fig.show()

def _on_clear(_):
    for row in _btns:
        for btn in row:
            btn.value = False
    with _pred_out:
        _pred_out.clear_output()

_predict_btn.on_click(_on_predict)
_clear_btn.on_click(_on_clear)

# ── Layout ───────────────────────────────────────────────────────────
_grid_ui = widgets.VBox(
    [widgets.HBox(row) for row in _btns]
    + [widgets.HBox([_predict_btn, _clear_btn])]
)
display(widgets.VBox([_grid_ui, _pred_out]))
